In [68]:
# IMPORTING NECESSARY LIBRARY
# imports the pandas library(used to manipulate and load tabular data), numpy for scientific computing, tensorflow for neural_network

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_curve,roc_auc_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [69]:
# LOADING DATA
X_train = pd.read_parquet("x_train_selected.parquet")
X_test  = pd.read_parquet("x_test_selected.parquet")
y_test  = pd.read_parquet("y_test.parquet").values.ravel()
y_train = pd.read_parquet("y_train.parquet").values.ravel()

# keep only normal samples
X_train_normal = X_train[y_train == 0]

# Convert labels to binary: 0 = normal, 1 = anomaly
y_test_bin = (y_test != 0).astype(int)
expected_anomalies = int(y_test_bin.sum())

In [70]:
#SCALE FEATURES
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled  = scaler.transform(X_test)

In [71]:
# ISOLATION FOREST

# Initializing the model
# n_estimators=200 → number of trees (more = more stable)
# contamination="auto"→ expected % of anomalies
# random_state=42 → reproducibility
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_train_scaled) # To train model only on feature data

train_iso_scores = -iso.decision_function(X_train_scaled)  # higher = more anomalous
test_iso_scores = -iso.decision_function(X_test_scaled)

# Choose threshold manually to match expected anomalies
threshold_iso = np.percentile(train_iso_scores, 95) # top N anomalies
iso_pred_bin = (test_iso_scores >= threshold_iso).astype(int)

print("Isolation Forest")
print("ROC-AUC:", roc_auc_score(y_test_bin, test_iso_scores))
print(classification_report(y_test_bin, iso_pred_bin))
fp_iso = ((iso_pred_bin == 1) & (y_test_bin == 0)).sum()
print("False Positives:", fp_iso, "\n")

Isolation Forest
ROC-AUC: 0.8387143320904898
              precision    recall  f1-score   support

           0       0.45      0.95      0.61     49601
           1       0.80      0.15      0.26     67298

    accuracy                           0.49    116899
   macro avg       0.63      0.55      0.43    116899
weighted avg       0.65      0.49      0.41    116899

False Positives: 2506 



In [72]:
# ONE-CLASS SVM

# kernel="rbf" → non-linear boundary
# gamma="scale" → automatic kernel scaling
# nu=0.05 → max fraction of anomalies
# It works well for complex distributions but slower
#ocsvm = OneClassSVM(
    #kernel="linear",
    #gamma="scale",
    #nu=0.05
#)

# Fits boundary around normal data
#ocsvm.fit(X_train_scaled)

# Distance from leared boundary ( more negative -> more anomalous)
#svm_scores = -ocsvm.decision_function(X_test_scaled)
#print(f"One-Class SVM score: {svm_scores}\n")

# Setting Threshold
#svm_threshold = np.percentile(svm_scores, 95)
#print(f"One-Class SVM threshold: {svm_threshold}\n")

# Converts predictions to anomaly labels
#svm_pred = ocsvm.predict(X_test_scaled)
#print(f"One-Class SVM predictions: {svm_pred}\n")

# Convert prediction to binary format
#y_pred = (svm_scores >= svm_threshold).astype(int)

# ONE-CLASS SVM - ROC_AUC and FALSEPOSITIVE ANALYSIS
#print("=== One-Class SVM ===")
#print("ROC-AUC:", roc_auc_score(y_test_bin, svm_scores))
#print(classification_report(y_test_bin, svm_pred))
#fp = ((svm_pred==1) & (y_test_bin==0)).sum()
#print("False Positives:", fp, "\n")

In [73]:
# AUTOENCODER
input_dim = X_train_scaled.shape[1]

# Enhanced Autoencoder
autoencoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),      # increase capacity
    layers.BatchNormalization(),               # stabilize training
    layers.Dropout(0.2),                       # reduce overfitting
    layers.Dense(64, activation="relu"),      # bottleneck
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dense(input_dim, activation="linear")
])

autoencoder.compile(optimizer="adam", loss="mse")

# EarlyStopping to avoid overfitting
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# Train
history = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    verbose=1,
    callbacks=[es]
)

# Reconstruction error
train_pred = autoencoder.predict(X_train_scaled)
train_error = np.mean((X_train_scaled - train_pred) ** 2, axis=1)

test_pred = autoencoder.predict(X_test_scaled)
test_error = np.mean((X_test_scaled - test_pred) ** 2, axis=1)

# Use ROC curve to pick threshold for FP < 2%
fpr, tpr, thresholds = roc_curve(y_test_bin, test_error)
# Choose threshold where fpr < 0.02
threshold_ae = thresholds[fpr < 0.02][-1]
ae_pred_bin = (test_error >= threshold_ae).astype(int)

print("Autoencoder - tuned")
print(classification_report(y_test_bin, ae_pred_bin))

fp_ae = ((ae_pred_bin == 1) & (y_test_bin == 0)).sum()
print("False Positives:", fp_ae)
print("ROC-AUC:", roc_auc_score(y_test_bin, test_error))

Epoch 1/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.2971 - val_loss: 0.0399
Epoch 2/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1589 - val_loss: 0.0358
Epoch 3/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1226 - val_loss: 0.0247
Epoch 4/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1023 - val_loss: 0.0396
Epoch 5/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0851 - val_loss: 0.0230
Epoch 6/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0744 - val_loss: 0.0359
Epoch 7/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0658 - val_loss: 0.0354
Epoch 8/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0592 - val_loss: 0.0238
Epoch 9/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0558 - val_loss: 0.0249
Epoch 10/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0529 - val_loss: 0.0208
Epoch 11/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0469 - val_loss: 0.0195
Epoch 12/100
698/698 ━━━━━━━━━━━━━━━━━━━━

In [74]:
# SUMMARY

print("Summary of false positives:")
print(f"Isolation Forest: {fp_iso}")
#print(f"One-Class SVM: {fp_svm}")
print(f"Autoencoder: {fp_ae}")

Summary of false positives:
Isolation Forest: 2506
Autoencoder: 991


In [75]:
# SAVE ISOLATION FOREST
import pickle

iso_artifacts = {
    "scaler": scaler,                    # fitted on normal training data
    "model": iso,                        # trained Isolation Forest
    "threshold": threshold_iso,          # decision threshold
    "feature_columns": X_train.columns.tolist()
}

with open("isolation_forest.pkl", "wb") as f:
    pickle.dump(iso_artifacts, f)

print("Isolation Forest saved successfully.")


Isolation Forest saved successfully.


In [76]:
# LOAD ISOLATION FOREST
with open("isolation_forest.pkl", "rb") as f:
    iso_artifacts = pickle.load(f)

scaler = iso_artifacts["scaler"]
iso = iso_artifacts["model"]
threshold_iso = iso_artifacts["threshold"]

In [77]:
# # SAVE AUTOENCODER

# autoencoder.save("autoencoder_model.keras")
# print("Autoencoder model saved.")


In [78]:
# # SAVE AUTOENCODER THRESHOLD

# ae_artifacts = {
#     "threshold": threshold_ae
# }

# with open("autoencoder_threshold.pkl", "wb") as f:
#     pickle.dump(ae_artifacts, f)

# print("Autoencoder threshold saved.")


# Create a dictionary to store all artifacts
ae_artifacts = {
    "scaler": scaler,                # fitted StandardScaler
    "model": autoencoder,            # trained autoencoder
    "threshold": threshold_ae,       # ROC-based threshold
    "feature_columns": X_train.columns.tolist()  # optional, for reference
}

# Save to a single pickle file
with open("autoencoder_full.pkl", "wb") as f:
    pickle.dump(ae_artifacts, f)

print("Autoencoder, scaler, and threshold saved together.")


Autoencoder, scaler, and threshold saved together.


In [79]:
# # LOAD AUTOENCODER

# from tensorflow.keras.models import load_model

# autoencoder = load_model("autoencoder_model.keras")

# with open("autoencoder_threshold.pkl", "rb") as f:
#     ae_artifacts = pickle.load(f)

# threshold_ae = ae_artifacts["threshold"]


# Load artifacts
with open("autoencoder_full.pkl", "rb") as f:
    ae_artifacts = pickle.load(f)

# Extract components
scaler = ae_artifacts["scaler"]
autoencoder = ae_artifacts["model"]
threshold_ae = ae_artifacts["threshold"]
feature_columns = ae_artifacts["feature_columns"]

print("Autoencoder, scaler, and threshold loaded successfully.")
print("Threshold:", threshold_ae)


Autoencoder, scaler, and threshold loaded successfully.
Threshold: 0.0710660611129874
